# <font color="#418FDE" size="6.5" uppercase>**Cross Decomposition**</font>

>Last update: 20260714.
    
By the end of this Lecture, you will be able to:
- Apply PLS and CCA methods to paired feature-target data. 
- Interpret latent scores, loadings, and predictions from cross-decomposition models. 
- Select component counts and scaling choices for cross-decomposition workflows. 


## **1. PLS Methods**

### **1.1. PLS Objectives**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_B/image_01_01.jpg?v=1784004084" width="250">



>* Find target-related patterns in paired data
>* Useful for many correlated predictors

>* Build latent scores linking paired datasets
>* Emphasize target-relevant patterns over noise

>* Stabilize regression with fewer target-focused components
>* Reduce complexity while preserving paired-data relationships



### **1.2. PLS Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_B/image_01_02.jpg?v=1784004082" width="250">



>* Predicts targets from many correlated inputs
>* Builds compact components for stable prediction

>* Find feature patterns linked to targets
>* Combine dimension reduction with supervised prediction

>* Match feature and target rows carefully
>* Use latent components to predict outcomes



In [ ]:
#@title Python Code - PLS Regression

# Demonstrate PLS regression on paired measurements.
# Latent components connect features with targets.
# Predictions improve using compact supervised scores.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Create correlated sensor features and one target.
rng = np.random.default_rng(42)
latent_quality = rng.normal(size=120)

# Each feature is a noisy view of the same hidden quality.
feature_weights = np.array([1.0, 0.9, 0.8, -0.7, -0.6, 0.5])
noise = rng.normal(scale=0.35, size=(120, 6))
X = latent_quality.reshape(-1, 1) * feature_weights + noise

y = 3.0 * latent_quality + rng.normal(scale=0.6, size=120)
y = y.reshape(-1, 1)

# Check that rows are paired observations.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature and target rows must match.")

# Split before scaling to avoid data leakage.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Scale features and target using training data only.
X_scaler = StandardScaler()
y_scaler = StandardScaler()
X_train_scaled = X_scaler.fit_transform(X_train)

X_test_scaled = X_scaler.transform(X_test)
y_train_scaled = y_scaler.fit_transform(y_train)

# Fit PLS with two supervised latent components.
pls = PLSRegression(n_components=2, scale=False)
pls.fit(X_train_scaled, y_train_scaled)

# Predict test targets and return to original units.
y_pred_scaled = pls.predict(X_test_scaled)
y_pred = y_scaler.inverse_transform(y_pred_scaled)

# Transform features into latent PLS scores.
train_scores = pls.transform(X_train_scaled)
test_r2 = r2_score(y_test, y_pred)

print("scikit-learn version:", sklearn.__version__)
print("Original feature count:", X.shape[1])
print("PLS component count:", pls.n_components)
print("Test R2:", round(test_r2, 3))
print("First training score pair:", np.round(train_scores[0], 3))

# Plot actual versus predicted target values.
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(y_test.ravel(), y_pred.ravel(), color="tab:blue", label="Test samples")

# Add a reference line for perfect predictions.
low = min(float(y_test.min()), float(y_pred.min()))
high = max(float(y_test.max()), float(y_pred.max()))
ax.plot([low, high], [low, high], color="black", linestyle="--", label="Perfect")

ax.set_title("PLS Regression: Actual vs Predicted Target")
ax.set_xlabel("Actual target")
ax.set_ylabel("Predicted target")
ax.legend()
plt.show()



### **1.3. PLS Canonical**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_B/image_01_03.jpg?v=1784004086" width="250">



>* Find shared patterns across paired data blocks
>* Use latent scores for noisy multivariate data

>* Create paired scores with strong shared covariance
>* Use weights to reveal contributing variables

>* Match and scale paired data carefully
>* Use scores and loadings to find patterns



In [ ]:
#@title Python Code - PLS Canonical

# This example demonstrates PLS Canonical analysis.
# Paired blocks share one hidden pattern.
# The plot shows matched latent scores.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.cross_decomposition import PLSCanonical
import sklearn

# A fixed generator makes the synthetic data reproducible.
rng = np.random.default_rng(42)
n_samples = 80

# One hidden factor influences both measured data blocks.
shared_factor = rng.normal(size=(n_samples, 1))
noise_x = rng.normal(scale=0.45, size=(n_samples, 4))
noise_y = rng.normal(scale=0.45, size=(n_samples, 3))

# Each block has different variables but matching rows.
x_weights_true = np.array([[1.0, 0.7, -0.5, 0.2]])
y_weights_true = np.array([[0.9, -0.6, 0.4]])

X = shared_factor @ x_weights_true + noise_x
Y = shared_factor @ y_weights_true + noise_y

# PLSCanonical finds paired scores with high covariance.
model = PLSCanonical(n_components=1, scale=True)
x_scores, y_scores = model.fit_transform(X, Y)

# Validate that the paired score arrays match the observations.
if x_scores.shape != (n_samples, 1) or y_scores.shape != (n_samples, 1):
    raise ValueError("Scores should have one row per observation.")

score_correlation = np.corrcoef(x_scores[:, 0], y_scores[:, 0])[0, 1]
print("scikit-learn version:", sklearn.__version__)
print("X block shape:", X.shape, "Y block shape:", Y.shape)
print("Correlation between paired latent scores:", round(score_correlation, 3))
print("First X-side weights:", np.round(model.x_weights_[:, 0], 2))
print("First Y-side weights:", np.round(model.y_weights_[:, 0], 2))

# The scatterplot shows how observations align in latent space.
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(x_scores[:, 0], y_scores[:, 0], alpha=0.75)
ax.set_title("PLS Canonical paired latent scores")
ax.set_xlabel("X block score, component 1")
ax.set_ylabel("Y block score, component 1")
ax.grid(True, alpha=0.3)
plt.show()



## **2. CCA Latent Views**

### **2.1. SVD PLS**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_B/image_02_01.jpg?v=1784004088" width="250">



>* Find paired directions that move together
>* Summarize feature-target links with latent scores

>* Scores locate observations; loadings explain variables
>* Paired views reveal shared measurement patterns

>* Predictions use learned latent patterns
>* Scores and loadings explain relationships



In [ ]:
#@title Python Code - SVD PLS

# This example demonstrates SVD PLS latent views.
# Scores summarize paired feature and target patterns.
# Loadings reveal which variables define each pattern.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler

# Synthetic paired data keeps the relationship easy to inspect.
rng = np.random.default_rng(42)
n_samples = 80
engagement = rng.normal(size=n_samples)

# Feature variables share one hidden engagement pattern.
attendance = 0.9 * engagement + rng.normal(scale=0.35, size=n_samples)
study_time = 0.8 * engagement + rng.normal(scale=0.35, size=n_samples)
participation = 0.7 * engagement + rng.normal(scale=0.35, size=n_samples)

# Target variables share a matching success pattern.
exam_score = 0.85 * engagement + rng.normal(scale=0.35, size=n_samples)
project_score = 0.75 * engagement + rng.normal(scale=0.35, size=n_samples)
satisfaction = 0.55 * engagement + rng.normal(scale=0.45, size=n_samples)

# PLS expects two numeric blocks with matching rows.
X = np.column_stack((attendance, study_time, participation))
Y = np.column_stack((exam_score, project_score, satisfaction))

if X.shape[0] != Y.shape[0]:
    raise ValueError("Feature and target blocks need matching rows.")

# Scaling makes loadings comparable across variables.
X_scaled = StandardScaler().fit_transform(X)
Y_scaled = StandardScaler().fit_transform(Y)

# One component finds the strongest shared latent direction.
pls = PLSRegression(n_components=1, scale=False)
pls.fit(X_scaled, Y_scaled)

# Scores are each observation's coordinates in the latent views.
X_scores, Y_scores = pls.transform(X_scaled, Y_scaled)
score_correlation = np.corrcoef(X_scores[:, 0], Y_scores[:, 0])[0, 1]

# Loadings show how original variables define the component.
feature_names = np.array(["attendance", "study_time", "participation"])
target_names = np.array(["exam", "project", "satisfaction"])
feature_loadings = np.round(pls.x_loadings_[:, 0], 2)
target_loadings = np.round(pls.y_loadings_[:, 0], 2)

print("scikit-learn version:", sklearn.__version__)
print("Latent score correlation:", round(score_correlation, 3))
print("Feature loadings:", dict(zip(feature_names, feature_loadings)))
print("Target loadings:", dict(zip(target_names, target_loadings)))

# The scatterplot shows paired latent views moving together.
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(X_scores[:, 0], Y_scores[:, 0], alpha=0.75)
ax.set_title("SVD PLS: paired latent scores")
ax.set_xlabel("Feature-side latent score")
ax.set_ylabel("Target-side latent score")
plt.show()



### **2.2. Fitting CCA Models**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_B/image_02_02.jpg?v=1784004090" width="250">



>* CCA creates paired summaries of two datasets
>* High score correlations reveal shared patterns

>* Weights build scores; loadings show associations
>* Interpret shared patterns, not isolated coefficients

>* Use CCA projections to explore shared structure
>* Check scaling, components, and score reliability



In [ ]:
#@title Python Code - Fitting CCA Models

# Fit CCA to paired synthetic data.
# Compare latent scores and variable loadings.
# Expect correlated views with interpretable patterns.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.cross_decomposition import CCA
from sklearn.preprocessing import StandardScaler

# Create two related data blocks from shared hidden factors.
rng = np.random.default_rng(42)
n_samples = 120
shared_factor = rng.normal(size=(n_samples, 2))

# Each block measures the shared factors differently.
x_noise = rng.normal(scale=0.35, size=(n_samples, 3))
y_noise = rng.normal(scale=0.35, size=(n_samples, 3))
X = shared_factor @ np.array([[1.0, 0.7, -0.2], [0.2, 0.8, 1.0]]) + x_noise
Y = shared_factor @ np.array([[0.9, -0.4, 0.6], [0.1, 1.0, 0.8]]) + y_noise

# Validate that the paired blocks have matching rows.
if X.shape[0] != Y.shape[0]:
    raise ValueError("X and Y must describe the same observations.")

# Scale each block so units do not dominate CCA.
X_scaled = StandardScaler().fit_transform(X)
Y_scaled = StandardScaler().fit_transform(Y)

# Fit two paired canonical components.
cca = CCA(n_components=2, scale=False, max_iter=500)
X_scores, Y_scores = cca.fit_transform(X_scaled, Y_scaled)

# Correlations summarize how strongly paired latent views align.
first_corr = np.corrcoef(X_scores[:, 0], Y_scores[:, 0])[0, 1]
second_corr = np.corrcoef(X_scores[:, 1], Y_scores[:, 1])[0, 1]

# Loadings connect original variables to the first latent score.
x_loadings = np.corrcoef(X_scaled.T, X_scores[:, 0])[:3, 3]
y_loadings = np.corrcoef(Y_scaled.T, Y_scores[:, 0])[:3, 3]

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Canonical correlations: {first_corr:.3f}, {second_corr:.3f}")
print(f"X loadings on component 1: {np.round(x_loadings, 2)}")
print(f"Y loadings on component 1: {np.round(y_loadings, 2)}")

# Plot the first paired scores for visual interpretation.
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(X_scores[:, 0], Y_scores[:, 0], alpha=0.75)
ax.set_title("CCA component 1: paired latent scores")
ax.set_xlabel("X-side canonical score")
ax.set_ylabel("Y-side canonical score")
plt.show()



### **2.3. Covariance Objectives**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_B/image_02_03.jpg?v=1784004092" width="250">



>* CCA finds paired views that move together
>* Components reflect shared variation, not isolated variance

>* Scores locate observations; loadings define patterns
>* Interpret signs through shared variable movement

>* Predict using shared patterns across data blocks
>* Validate latent views with context and judgment



## **3. Component Practice**

### **3.1. Scaling Paired Data**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_B/image_03_01.jpg?v=1784004077" width="250">



>* Scale variables to prevent unfair dominance
>* Center and standardize for balanced contributions

>* Scale different data blocks before modeling
>* Avoid unit artifacts in shared components

>* Fit scaling on training data only
>* Match scaling to the modeling goal



In [ ]:
#@title Python Code - Scaling Paired Data

# This example compares scaling choices for paired data.
# PLS components can change when units dominate.
# The output shows safer train-only scaling practice.

import numpy as np
import sklearn
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# A fixed generator makes the synthetic paired data repeatable.
rng = np.random.default_rng(42)
n_samples = 120
shared_signal = rng.normal(size=n_samples)

# Feature columns use very different numerical scales.
small_scale_feature = shared_signal + rng.normal(scale=0.4, size=n_samples)
large_scale_noise = rng.normal(scale=80.0, size=n_samples)
medium_scale_feature = 5.0 * shared_signal + rng.normal(scale=1.0, size=n_samples)

# Targets also share the signal but use different units.
quality_score = 2.0 * shared_signal + rng.normal(scale=0.5, size=n_samples)
lab_measurement = 50.0 * shared_signal + rng.normal(scale=10.0, size=n_samples)

X = np.column_stack((small_scale_feature, large_scale_noise, medium_scale_feature))
Y = np.column_stack((quality_score, lab_measurement))

# Validate the paired blocks before modeling.
if X.shape != (n_samples, 3) or Y.shape != (n_samples, 2):
    raise ValueError("The paired data blocks have unexpected shapes.")

# Split first so scaling is learned only from training rows.
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.30, random_state=42
)

# PLSRegression with scale=False preserves raw unit differences.
raw_pls = PLSRegression(n_components=1, scale=False)
raw_pls.fit(X_train, Y_train)
raw_r2 = r2_score(Y_test, raw_pls.predict(X_test))

# StandardScaler is fit inside the pipeline on training data only.
scaled_pls = make_pipeline(
    StandardScaler(), PLSRegression(n_components=1, scale=True)
)
scaled_pls.fit(X_train, Y_train)
scaled_r2 = r2_score(Y_test, scaled_pls.predict(X_test))

# Compare which feature dominates the first latent component.
raw_weights = np.abs(raw_pls.x_weights_[:, 0])
scaled_weights = np.abs(scaled_pls.named_steps["plsregression"].x_weights_[:, 0])

feature_names = ["small signal", "large noise", "medium signal"]
raw_top = feature_names[int(np.argmax(raw_weights))]
scaled_top = feature_names[int(np.argmax(scaled_weights))]

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Raw-unit PLS test R2: {raw_r2:.3f}")
print(f"Train-scaled PLS test R2: {scaled_r2:.3f}")
print(f"Largest raw-unit component weight: {raw_top}")
print(f"Largest scaled component weight: {scaled_top}")



### **3.2. Choosing Components**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_B/image_03_02.jpg?v=1784004079" width="250">



>* Keep shared patterns, not random noise
>* Choose components that predict and explain

>* Use validation to choose component counts
>* Prefer simple models with stable performance

>* Favor interpretable components over maximum complexity
>* Balance accuracy, stability, simplicity, and meaning



In [ ]:
#@title Python Code - Choosing Components

# This example chooses PLS component counts.
# Cross-validation estimates held-out prediction error.
# The plot favors simple stable models.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import sklearn

# Synthetic paired blocks share three real latent patterns.
rng = np.random.default_rng(42)
n_samples = 120
n_latent = 3

latent_scores = rng.normal(size=(n_samples, n_latent))
x_weights = rng.normal(size=(n_latent, 10))
y_weights = rng.normal(size=(n_latent, 2))

# Extra noise makes too many components less useful.
X = latent_scores @ x_weights + rng.normal(scale=0.7, size=(n_samples, 10))
Y = latent_scores @ y_weights + rng.normal(scale=0.7, size=(n_samples, 2))

if X.shape[0] != Y.shape[0]:
    raise ValueError("X and Y must contain the same paired rows.")

component_counts = [1, 2, 3, 4, 5, 6]
cv = KFold(n_splits=5, shuffle=True, random_state=42)
mean_errors = []

# Each fold fits scaling inside PLS on training data only.
for n_components in component_counts:
    fold_errors = []
    for train_index, test_index in cv.split(X):
        model = PLSRegression(n_components=n_components, scale=True)
        model.fit(X[train_index], Y[train_index])
        predictions = model.predict(X[test_index])
        fold_errors.append(mean_squared_error(Y[test_index], predictions))
    mean_errors.append(float(np.mean(fold_errors)))

best_index = int(np.argmin(mean_errors))
best_components = component_counts[best_index]
best_error = mean_errors[best_index]

print("scikit-learn version:", sklearn.__version__)
print("Best component count:", best_components)
print("Best mean CV MSE:", round(best_error, 3))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(component_counts, mean_errors, marker="o", label="5-fold CV MSE")
ax.axvline(best_components, color="red", linestyle="--", label="chosen count")
ax.set_title("Choosing PLS Components by Cross-Validation")
ax.set_xlabel("Number of PLS components")
ax.set_ylabel("Mean squared prediction error")
ax.legend()
plt.show()



### **3.3. Reading Loadings**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_B/image_03_03.jpg?v=1784004080" width="250">



>* Loadings reveal variables tied to components
>* They turn latent scores into meaning

>* Compare loading size and relative direction
>* Standardize data before judging variable importance

>* Use loadings to judge component usefulness
>* Confirm patterns with validation and domain knowledge



In [ ]:
#@title Python Code - Reading Loadings

# This example reads PLS loadings.
# Scaling changes which variables dominate components.
# The output compares raw and scaled interpretations.

import numpy as np
import pandas as pd
import sklearn
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler

# Create paired feature and target blocks with different units.
rng = np.random.default_rng(42)
n_samples = 80

activity = rng.normal(0, 1, n_samples)
health = rng.normal(0, 1, n_samples)
noise = rng.normal(0, 0.4, (n_samples, 4))

# Income has a much larger numeric scale than ratings.
features = pd.DataFrame(
    {
        "income_dollars": 60000 + 18000 * activity + 3000 * noise[:, 0],
        "site_visits": 20 + 6 * activity + noise[:, 1],
        "survey_rating": 3 + 0.7 * health + 0.2 * noise[:, 2],
        "support_calls": 5 - 1.2 * health + noise[:, 3],
    }
)

targets = pd.DataFrame(
    {
        "future_spend": 200 + 45 * activity + rng.normal(0, 8, n_samples),
        "satisfaction": 70 + 12 * health + rng.normal(0, 3, n_samples),
    }
)

# Validate the paired blocks before fitting the model.
if features.shape[0] != targets.shape[0]:
    raise ValueError("Feature and target rows must match.")

print("scikit-learn version:", sklearn.__version__)
print("Rows and columns:", features.shape[0], "rows,", features.shape[1], "features")

# Fit PLS without scaling to show unit dominance.
raw_pls = PLSRegression(n_components=2, scale=False)
raw_pls.fit(features, targets)

# Fit PLS after standardizing both paired data blocks.
x_scaler = StandardScaler()
y_scaler = StandardScaler()
scaled_x = x_scaler.fit_transform(features)
scaled_y = y_scaler.fit_transform(targets)

scaled_pls = PLSRegression(n_components=2, scale=False)
scaled_pls.fit(scaled_x, scaled_y)

# Put component loadings into small readable tables.
raw_loadings = pd.DataFrame(
    raw_pls.x_loadings_, index=features.columns, columns=["comp1", "comp2"]
)
scaled_loadings = pd.DataFrame(
    scaled_pls.x_loadings_, index=features.columns, columns=["comp1", "comp2"]
)

print("Raw component 1 loadings:")
print(raw_loadings[["comp1"]].round(3).T)
print("Scaled component 1 loadings:")
print(scaled_loadings[["comp1"]].round(3).T)

# Identify the strongest variable after fair scaling.
strongest = scaled_loadings["comp1"].abs().idxmax()
value = scaled_loadings.loc[strongest, "comp1"]
print("Strongest scaled component 1 feature:", strongest, "=", round(value, 3))



# <font color="#418FDE" size="6.5" uppercase>**Cross Decomposition**</font>


In this lecture, you learned to:
- Apply PLS and CCA methods to paired feature-target data. 
- Interpret latent scores, loadings, and predictions from cross-decomposition models. 
- Select component counts and scaling choices for cross-decomposition workflows. 

In the next Module (Module 15), we will go over 'Bayes Calibration'